In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Brain-Mets-Lung-MRI-Path-Segs**

## Radiomic Features

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor

# ============ CONFIG ============
LUNG_ROOT = "/content/drive/MyDrive/Datasets/Brain-Mets-Lung-MRI-Path-Segs"
OUT_CSV = "/content/drive/MyDrive/lung_radiomics_from_images.csv"

# ============ EXTRACTOR ============
extractor = featureextractor.RadiomicsFeatureExtractor()

# ============ HELPERS ============
def get_nonzero_labels(mask_img):
    arr = sitk.GetArrayFromImage(mask_img)
    labels = np.unique(arr)
    labels = labels[labels != 0]
    return labels

def geometries_match(img, msk):
    return (
        img.GetSize() == msk.GetSize() and
        img.GetSpacing() == msk.GetSpacing() and
        img.GetOrigin() == msk.GetOrigin() and
        img.GetDirection() == msk.GetDirection()
    )

def resample_mask_to_image(mask, image):
    """
    Resample mask to match image geometry.
    Nearest neighbor interpolation is used because this is a segmentation mask.
    """
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(image)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampler.SetDefaultPixelValue(0)
    return resampler.Execute(mask)

def extract_features(image_path, mask_path, prefix):
    """
    Returns dict of radiomics features, prefixed.
    If extraction fails or ROI is empty -> returns {}.
    Automatically resamples mask only if geometry does not match.
    """
    try:
        img = sitk.ReadImage(image_path)
        msk = sitk.ReadImage(mask_path)

        # Fix only misaligned cases
        if not geometries_match(img, msk):
            print(f"Resampling mask for: {os.path.basename(image_path)}")
            msk = resample_mask_to_image(msk, img)

        labels = get_nonzero_labels(msk)
        if len(labels) == 0:
            print(f"Empty ROI for: {os.path.basename(image_path)}")
            return {}

        # Use first non-zero label
        extractor.settings["label"] = int(labels[0])

        feats = extractor.execute(img, msk)
        feats = {
            f"{prefix}{k}": v
            for k, v in feats.items()
            if not k.startswith("diagnostics_")
        }
        return feats

    except Exception as e:
        print(f"Failed on {os.path.basename(image_path)}: {e}")
        return {}

def discover_accessions(patient_path):
    """
    Finds unique accessions by looking for files like:
      <accession>_t1ce_img.nii.gz
      <accession>_flair_img.nii.gz
      <accession>_core_seg.nii.gz
      <accession>_whole_seg.nii.gz
    Returns a sorted list of accession strings.
    """
    accessions = set()
    for fn in os.listdir(patient_path):
        m = re.match(r"(.+?)_(t1ce_img|flair_img|core_seg|whole_seg)\.nii\.gz$", fn)
        if m:
            accessions.add(m.group(1))
    return sorted(accessions)

# ============ MAIN LOOP ============
rows = []

patient_ids = sorted([
    d for d in os.listdir(LUNG_ROOT)
    if os.path.isdir(os.path.join(LUNG_ROOT, d))
])

print("Patients found:", len(patient_ids))

for patient_id in patient_ids:
    patient_path = os.path.join(LUNG_ROOT, patient_id)
    accessions = discover_accessions(patient_path)

    if not accessions:
        continue

    for acc in accessions:
        # Build expected paths
        t1ce_img = os.path.join(patient_path, f"{acc}_t1ce_img.nii.gz")
        core_seg = os.path.join(patient_path, f"{acc}_core_seg.nii.gz")
        flair_img = os.path.join(patient_path, f"{acc}_flair_img.nii.gz")
        whole_seg = os.path.join(patient_path, f"{acc}_whole_seg.nii.gz")

        row = {"patient_ID": patient_id, "accession": acc}

        # T1CE + CORE
        if os.path.exists(t1ce_img) and os.path.exists(core_seg):
            row.update(extract_features(t1ce_img, core_seg, prefix="t1ce__"))

        # FLAIR + WHOLE
        if os.path.exists(flair_img) and os.path.exists(whole_seg):
            row.update(extract_features(flair_img, whole_seg, prefix="flair__"))

        rows.append(row)

# ============ SAVE OUTPUT ============
df_out = pd.DataFrame(rows)

print("Rows (accessions):", df_out.shape[0])
print("Columns:", df_out.shape[1])
display(df_out.head(3))

df_out.to_csv(OUT_CSV, index=False)
print("✅ Saved to:", OUT_CSV)

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}


Patients found: 103


INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask


Resampling mask for: YG_XKGYFGZHUMTO_t1ce_img.nii.gz


INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Computing shape


...

Resampling mask for: YG_OUD88SBAPAOC_flair_img.nii.gz


INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm


Rows (accessions): 111
Columns: 216


,patient_ID,accession,t1ce__original_shape_Elongation,t1ce__original_shape_Flatness,t1ce__original_shape_LeastAxisLength,t1ce__original_shape_MajorAxisLength,t1ce__original_shape_Maximum2DDiameterColumn,t1ce__original_shape_Maximum2DDiameterRow,t1ce__original_shape_Maximum2DDiameterSlice,t1ce__original_shape_Maximum3DDiameter,...,flair__original_glszm_SmallAreaHighGrayLevelEmphasis,flair__original_glszm_SmallAreaLowGrayLevelEmphasis,flair__original_glszm_ZoneEntropy,flair__original_glszm_ZonePercentage,flair__original_glszm_ZoneVariance,flair__original_ngtdm_Busyness,flair__original_ngtdm_Coarseness,flair__original_ngtdm_Complexity,flair__original_ngtdm_Contrast,flair__original_ngtdm_Strength
0,YG_0427RC24FT6W,YG_XKGYFGZHUMTO,0.881850,0.577758,26.327191,45.567853,48.9716790035904,56.633453479181135,54.45005786413758,59.77462140566726,...,59.93124490461693,0.00977572292386392,5.875251843214384,0.07839262187088274,3428.478426664784,3.151025145457232,0.0012599692437947321,135.5280532797845,0.02472108265201985,0.1974614246083836
1,YG_0AXGKD8AFJGS,YG_HAEAKFOSOSWC,0.883262,0.652214,22.187268,34.018365,39.66106403010388,36.22154055254967,37.0,41.49698784249286,...,19.26907597529036,0.02359211806621709,5.572031590613505,0.01489999552552687,386003.87886986084,16.502954212318098,0.0004567273888351139,28.616244039980725,0.012254787623053251,0.04085112092023748
2,YG_0IBUXTBINCD9,YG_985XO5NHL516,0.317696,0.246274,29.824431,121.102451,89.64769288654746,94.40753106371213,47.34521247273733,96.29848569870204,...,117.36473645790862,0.004149651804415331,6.633930650649914,0.08694843539312781,7406.972993508537,4.945764101253929,0.0005002810223189947,323.518459227843,0.05823301267246309,0.12406124713334385


✅ Saved to: /content/drive/MyDrive/lung_radiomics_from_images.csv


## Preprocessing

### Cropping the Lung cancer origin dataset

In [ ]:
!pip install SimpleITK
import os
import re
import numpy as np
import SimpleITK as sitk

# =========================
# CONFIG
# =========================
ROOT = "/content/drive/MyDrive/Datasets/Brain-Mets-Lung-MRI-Path-Segs"
OUT_DIR = "/content/drive/MyDrive/Lung_Cropped_ROI"

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# GEOMETRY FUNCTIONS
# =========================
def geometries_match(img, msk):
    return (
        img.GetSize() == msk.GetSize() and
        img.GetSpacing() == msk.GetSpacing() and
        img.GetOrigin() == msk.GetOrigin() and
        img.GetDirection() == msk.GetDirection()
    )

def resample_mask_to_image(mask, image):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(image)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampler.SetDefaultPixelValue(0)
    return resampler.Execute(mask)

# =========================
# ACCESSION DISCOVERY
# =========================
def discover_accessions(patient_path):
    accessions = set()

    for fn in os.listdir(patient_path):
        m = re.match(r"(.+?)_(t1ce_img|flair_img|core_seg|whole_seg)\.nii\.gz$", fn)
        if m:
            accessions.add(m.group(1))

    return sorted(accessions)

# =========================
# PATIENT LIST
# =========================
patient_ids = sorted([
    d for d in os.listdir(ROOT)
    if os.path.isdir(os.path.join(ROOT, d))
])

print("Total patients:", len(patient_ids))

# =========================
# MAIN LOOP
# =========================
for patient_id in patient_ids:

    patient_path = os.path.join(ROOT, patient_id)
    accessions = discover_accessions(patient_path)

    if not accessions:
        continue

    print(f"\nProcessing patient: {patient_id}")

    for acc in accessions:

        print(f"  Accession: {acc}")

        # ==================================================
        # T1CE + CORE SEGMENTATION
        # ==================================================
        t1ce_path = os.path.join(patient_path, f"{acc}_t1ce_img.nii.gz")
        core_path = os.path.join(patient_path, f"{acc}_core_seg.nii.gz")

        if os.path.exists(t1ce_path) and os.path.exists(core_path):

            img = sitk.ReadImage(t1ce_path)
            seg = sitk.ReadImage(core_path)

            # Fix geometry mismatch
            if not geometries_match(img, seg):
                print("    Resampling CORE mask")
                seg = resample_mask_to_image(seg, img)

            img_arr = sitk.GetArrayFromImage(img).astype(np.float32)
            seg_arr = sitk.GetArrayFromImage(seg)

            binary_mask = (seg_arr > 0).astype(np.float32)

            coords = np.argwhere(binary_mask > 0)

            if coords.size > 0:

                zmin, ymin, xmin = coords.min(axis=0)
                zmax, ymax, xmax = coords.max(axis=0)

                roi_masked = img_arr * binary_mask
                roi = roi_masked[zmin:zmax+1, ymin:ymax+1, xmin:xmax+1]

                out_file = os.path.join(
                    OUT_DIR,
                    f"{patient_id}_{acc}_t1ce_roi.npy"
                )

                np.save(out_file, roi)

                print("    Saved T1CE ROI:", roi.shape)

                del img, seg, img_arr, seg_arr, binary_mask, coords, roi_masked, roi

        # ==================================================
        # FLAIR + WHOLE SEGMENTATION
        # ==================================================
        flair_path = os.path.join(patient_path, f"{acc}_flair_img.nii.gz")
        whole_path = os.path.join(patient_path, f"{acc}_whole_seg.nii.gz")

        if os.path.exists(flair_path) and os.path.exists(whole_path):

            img = sitk.ReadImage(flair_path)
            seg = sitk.ReadImage(whole_path)

            # Fix geometry mismatch
            if not geometries_match(img, seg):
                print("    Resampling WHOLE mask")
                seg = resample_mask_to_image(seg, img)

            img_arr = sitk.GetArrayFromImage(img).astype(np.float32)
            seg_arr = sitk.GetArrayFromImage(seg)

            binary_mask = (seg_arr > 0).astype(np.float32)

            coords = np.argwhere(binary_mask > 0)

            if coords.size > 0:

                zmin, ymin, xmin = coords.min(axis=0)
                zmax, ymax, xmax = coords.max(axis=0)

                roi_masked = img_arr * binary_mask
                roi = roi_masked[zmin:zmax+1, ymin:ymax+1, xmin:xmax+1]

                out_file = os.path.join(
                    OUT_DIR,
                    f"{patient_id}_{acc}_flair_roi.npy"
                )

                np.save(out_file, roi)

                print("    Saved FLAIR ROI:", roi.shape)

                del img, seg, img_arr, seg_arr, binary_mask, coords, roi_masked, roi

print("\nAll patients processed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 12.4 MB/s eta 0:00:00
Total patients: 103

Processing patient: YG_0427RC24FT6W
  Accession: YG_XKGYFGZHUMTO
    Resampling CORE mask
    Saved T1CE ROI: (5, 58, 52)
    Resampling WHOLE mask
    Saved FLAIR ROI: (6, 64, 64)

Processing patient: YG_0AXGKD8AFJGS
  Accession: YG_HAEAKFOSOSWC
    Saved T1CE ROI: (28, 36, 39)
    Resampling WHOLE mask
    Saved FLAIR ROI: (24, 53, 60)

Processing patient: YG_0IBUXTBINCD9
  Accession: YG_985XO5NHL516
    Resampling CORE mask
    Saved T1CE ROI: (101, 60, 49)
    Resampling WHOLE mask
    Saved FLAIR ROI: (21, 70, 87)

Processing patient: YG_0LCI9NV44UK0
  Accession: YG_WNJIJZXQW7BT
    Saved T1CE ROI: (5, 11, 10)
    Resampling WHOLE mask
    Saved FLAIR ROI: (5, 65, 47)

Processing patient: YG_1BZ1NP805Q9J
  Accession: YG_3ULZIC6OE5NB
    Resampling CORE mask
    Saved T1CE ROI: (6, 36, 63)
    Resampling WHOLE mask
    Saved FLAIR ROI: (3, 22, 12)

Processing patient: YG_1PHFOPJWEXFN

Checking Images

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

OUT_DIR = "/content/drive/MyDrive/Lung_Cropped_ROI"

patient_id = "YG_2XR67085OZD3"
accession  = "YG_PT6LAWSXHST3"
seq = "t1ce"

file_path = os.path.join(OUT_DIR, f"{patient_id}_{accession}_{seq}_roi.npy")
roi = np.load(file_path)

def show_slice(z):
    plt.figure(figsize=(6, 6))
    plt.imshow(roi[z], cmap="gray", interpolation="nearest")
    plt.title(f"{patient_id} | {accession} | {seq} | slice {z}")
    plt.axis("off")
    plt.show()

interact(
    show_slice,
    z=IntSlider(min=0, max=roi.shape[0]-1, step=1, value=roi.shape[0]//2)
);

interactive(children=(IntSlider(value=4, description='z', max=7), Output()), _dom_classes=('widget-interact',)…

### Resizing Images to 256x256

In [ ]:
import os
import numpy as np
import scipy.ndimage as ndi

# =========================
# CONFIG
# =========================
IN_DIR = "/content/drive/MyDrive/Lung_Cropped_ROI"
OUT_DIR = "/content/drive/MyDrive/Lung_Resized_ROI"
os.makedirs(OUT_DIR, exist_ok=True)

target_xy = (256, 256)   # resize only Y, X

# =========================
# GET ALL ROI FILES
# =========================
roi_files = sorted([f for f in os.listdir(IN_DIR) if f.endswith(".npy")])

print("Total ROI files found:", len(roi_files))

# =========================
# LOOP THROUGH FILES
# =========================
for file_name in roi_files:

    in_path = os.path.join(IN_DIR, file_name)
    out_path = os.path.join(OUT_DIR, file_name.replace("_roi.npy", "_resized.npy"))

    try:
        roi = np.load(in_path).astype(np.float32)

        # Skip empty arrays
        if roi.size == 0:
            print(f"Skipped empty file: {file_name}")
            continue

        print(f"\nProcessing: {file_name}")
        print("Original shape:", roi.shape)

        # Keep Z slices unchanged
        zoom_factors = (
            1,
            target_xy[0] / roi.shape[1],
            target_xy[1] / roi.shape[2]
        )

        # Resize with interpolation
        roi_resized = ndi.zoom(roi, zoom_factors, order=1)

        # Save
        np.save(out_path, roi_resized)

        print("Resized shape:", roi_resized.shape)
        print(f"Saved: {out_path}")

        del roi, roi_resized

    except Exception as e:
        print(f"Failed on {file_name}: {e}")

Total ROI files found: 211

Processing: YG_0427RC24FT6W_YG_XKGYFGZHUMTO_flair_roi.npy
Original shape: (6, 64, 64)
Resized shape: (6, 256, 256)
Saved: /content/drive/MyDrive/Lung_Resized_ROI/YG_0427RC24FT6W_YG_XKGYFGZHUMTO_flair_resized.npy

Processing: YG_0427RC24FT6W_YG_XKGYFGZHUMTO_t1ce_roi.npy
Original shape: (5, 58, 52)
Resized shape: (5, 256, 256)
Saved: /content/drive/MyDrive/Lung_Resized_ROI/YG_0427RC24FT6W_YG_XKGYFGZHUMTO_t1ce_resized.npy

Processing: YG_0AXGKD8AFJGS_YG_HAEAKFOSOSWC_flair_roi.npy
Original shape: (24, 53, 60)
Resized shape: (24, 256, 256)
Saved: /content/drive/MyDrive/Lung_Resized_ROI/YG_0AXGKD8AFJGS_YG_HAEAKFOSOSWC_flair_resized.npy

Processing: YG_0AXGKD8AFJGS_YG_HAEAKFOSOSWC_t1ce_roi.npy
Original shape: (28, 36, 39)
Resized shape: (28, 256, 256)
Saved: /content/drive/MyDrive/Lung_Resized_ROI/YG_0AXGKD8AFJGS_YG_HAEAKFOSOSWC_t1ce_resized.npy

Processing: YG_0IBUXTBINCD9_YG_985XO5NHL516_flair_roi.npy
Original shape: (21, 70, 87)
Resized shape: (21, 256, 256)
Sav

Checking Images

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# =========================
# CONFIG
# =========================
patient_id = "YG_2XR67085OZD3"
accession  = "YG_PT6LAWSXHST3"
seqs = ["t1ce", "flair"]

cropped_dir = "/content/drive/MyDrive/Lung_Cropped_ROI"
resized_dir = "/content/drive/MyDrive/Lung_Resized_ROI"

# =========================
# LOAD ALL SEQUENCES
# =========================
original_data = {}
resized_data = {}

for seq in seqs:
    original_path = os.path.join(cropped_dir, f"{patient_id}_{accession}_{seq}_roi.npy")
    resized_path  = os.path.join(resized_dir, f"{patient_id}_{accession}_{seq}_resized.npy")

    original_data[seq] = np.load(original_path) if os.path.exists(original_path) else None
    resized_data[seq]  = np.load(resized_path) if os.path.exists(resized_path) else None

# =========================
# DISPLAY FUNCTION
# =========================
def show_all(z_percent):
    fig, ax = plt.subplots(2, 2, figsize=(10, 8))

    for i, seq in enumerate(seqs):

        # ---- original row
        if original_data[seq] is not None:
            z_orig = int((z_percent / 100) * (original_data[seq].shape[0] - 1))
            ax[0, i].imshow(original_data[seq][z_orig], cmap="gray", interpolation="nearest")
            ax[0, i].set_title(f"{seq} original\nslice {z_orig} | {original_data[seq].shape}")
        else:
            ax[0, i].set_title(f"{seq} original\nmissing")
        ax[0, i].axis("off")

        # ---- resized row
        if resized_data[seq] is not None:
            z_res = int((z_percent / 100) * (resized_data[seq].shape[0] - 1))
            ax[1, i].imshow(resized_data[seq][z_res], cmap="gray", interpolation="nearest")
            ax[1, i].set_title(f"{seq} resized\nslice {z_res} | {resized_data[seq].shape}")
        else:
            ax[1, i].set_title(f"{seq} resized\nmissing")
        ax[1, i].axis("off")

    plt.suptitle(f"{patient_id} | {accession} | relative slice {z_percent}%", fontsize=14)
    plt.tight_layout()
    plt.show()

interact(
    show_all,
    z_percent=IntSlider(min=0, max=100, step=1, value=50)
);

interactive(children=(IntSlider(value=50, description='z_percent'), Output()), _dom_classes=('widget-interact'…

### Normalization

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import numpy as np
import shutil

# =========================
# CONFIG
# =========================
IN_DIR  = "/content/drive/MyDrive/Lung_Resized_ROI"
OUT_DIR = "/content/drive/MyDrive/Lung_Normalized_ROI"
TMP_DIR = "/content/_tmp_norm_lung"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

BAD_LOG_LOCAL = "/content/_bad_lung_files.txt"
BAD_LOG_DRIVE = os.path.join(OUT_DIR, "_bad_lung_files.txt")

# =========================
# HELPERS
# =========================
def is_drive_disconnected(e):
    msg = str(e)
    return (getattr(e, "errno", None) == 107) or ("Transport endpoint is not connected" in msg)

def remount_drive():
    print("🔄 Drive disconnected → remounting...")
    drive.mount('/content/drive', force_remount=True)

def minmax_normalize(arr, eps=1e-8):
    arr = arr.astype(np.float32)
    mn = arr.min()
    mx = arr.max()
    if (mx - mn) < eps:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - mn) / (mx - mn)

def safe_load(path, retries=3):
    for _ in range(retries):
        try:
            return np.load(path)
        except OSError as e:
            if is_drive_disconnected(e):
                remount_drive()
                continue
            raise
    raise OSError(f"Failed to load after {retries} retries: {path}")

def safe_copy(local_path, drive_path, retries=3):
    for _ in range(retries):
        try:
            shutil.copy2(local_path, drive_path)
            return
        except OSError as e:
            if is_drive_disconnected(e):
                remount_drive()
                continue
            raise
    raise OSError(f"Failed to copy after {retries} retries: {local_path} -> {drive_path}")

# =========================
# FILE LIST
# =========================
files = sorted([f for f in os.listdir(IN_DIR) if f.endswith(".npy")])
print("✅ Found .npy files:", len(files))

bad = []
saved = 0

# =========================
# NORMALIZE LOOP
# =========================
for i, fname in enumerate(files, 1):
    in_path  = os.path.join(IN_DIR, fname)
    out_path = os.path.join(OUT_DIR, fname)

    try:
        x = safe_load(in_path)
        x_norm = minmax_normalize(x)

        tmp_out = os.path.join(TMP_DIR, fname)
        np.save(tmp_out, x_norm)      # save locally first
        safe_copy(tmp_out, out_path)  # then copy to Drive
        os.remove(tmp_out)

        saved += 1
        print(f"[{i}/{len(files)}] ✅ Saved {fname} | shape={x.shape} | min={x_norm.min():.3f} max={x_norm.max():.3f}")

    except ValueError as e:
        bad.append(f"{fname} | ValueError: {e}")
        print(f"[{i}/{len(files)}] ⚠️ Corrupted .npy (ValueError): {fname}")

    except Exception as e:
        bad.append(f"{fname} | {type(e).__name__}: {e}")
        print(f"[{i}/{len(files)}] ⚠️ Skipped (I/O error): {fname} ({type(e).__name__})")

# =========================
# WRITE LOG SAFELY
# =========================
with open(BAD_LOG_LOCAL, "w") as f:
    f.write("\n".join(bad))

print("\n✅ Finished normalization.")
print("Saved files:", saved)
print("Bad/Skipped files:", len(bad))
print("Local bad log:", BAD_LOG_LOCAL)

try:
    safe_copy(BAD_LOG_LOCAL, BAD_LOG_DRIVE)
    print("✅ Bad log copied to Drive:", BAD_LOG_DRIVE)
except Exception as e:
    print("⚠️ Could not copy bad log to Drive. Keep local log:", BAD_LOG_LOCAL)
    print("Reason:", e)

Mounted at /content/drive
✅ Found .npy files: 211
[1/211] ✅ Saved YG_0427RC24FT6W_YG_XKGYFGZHUMTO_flair_resized.npy | shape=(6, 256, 256) | min=0.000 max=1.000
[2/211] ✅ Saved YG_0427RC24FT6W_YG_XKGYFGZHUMTO_t1ce_resized.npy | shape=(5, 256, 256) | min=0.000 max=1.000
[3/211] ✅ Saved YG_0AXGKD8AFJGS_YG_HAEAKFOSOSWC_flair_resized.npy | shape=(24, 256, 256) | min=0.000 max=1.000
[4/211] ✅ Saved YG_0AXGKD8AFJGS_YG_HAEAKFOSOSWC_t1ce_resized.npy | shape=(28, 256, 256) | min=0.000 max=1.000
[5/211] ✅ Saved YG_0IBUXTBINCD9_YG_985XO5NHL516_flair_resized.npy | shape=(21, 256, 256) | min=0.000 max=1.000
[6/211] ✅ Saved YG_0IBUXTBINCD9_YG_985XO5NHL516_t1ce_resized.npy | shape=(101, 256, 256) | min=0.000 max=1.000
[7/211] ✅ Saved YG_0LCI9NV44UK0_YG_WNJIJZXQW7BT_flair_resized.npy | shape=(5, 256, 256) | min=0.000 max=1.000
[8/211] ✅ Saved YG_0LCI9NV44UK0_YG_WNJIJZXQW7BT_t1ce_resized.npy | shape=(5, 256, 256) | min=0.000 max=1.000
[9/211] ✅ Saved YG_1BZ1NP805Q9J_YG_3ULZIC6OE5NB_flair_resized.npy | s

Checking images

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# =========================
# CONFIG
# =========================
patient_id = "YG_2XR67085OZD3"      # change
accession  = "YG_PT6LAWSXHST3"      # change
seqs = ["t1ce", "flair"]

orig_dir = "/content/drive/MyDrive/Lung_Resized_ROI"
norm_dir = "/content/drive/MyDrive/Lung_Normalized_ROI"

# =========================
# LOAD DATA
# =========================
orig_data = {}
norm_data = {}

for seq in seqs:
    orig_path = os.path.join(orig_dir, f"{patient_id}_{accession}_{seq}_resized.npy")
    norm_path = os.path.join(norm_dir, f"{patient_id}_{accession}_{seq}_resized.npy")

    orig_data[seq] = np.load(orig_path) if os.path.exists(orig_path) else None
    norm_data[seq] = np.load(norm_path) if os.path.exists(norm_path) else None

# =========================
# FIND MAX DEPTH
# =========================
depths = []

for seq in seqs:
    if orig_data[seq] is not None:
        depths.append(orig_data[seq].shape[0])
    if norm_data[seq] is not None:
        depths.append(norm_data[seq].shape[0])

max_depth = max(depths)

# =========================
# DISPLAY FUNCTION
# =========================
def show_slice(z):
    fig, ax = plt.subplots(2, 2, figsize=(10, 8))

    for i, seq in enumerate(seqs):

        # ---- original resized
        if orig_data[seq] is not None and z < orig_data[seq].shape[0]:
            img = orig_data[seq][z]
            ax[0, i].imshow(img, cmap="gray", interpolation="nearest")
            ax[0, i].set_title(
                f"{seq} resized\n"
                f"min={img.min():.3f} max={img.max():.3f}\n"
                f"{orig_data[seq].shape}"
            )
        else:
            ax[0, i].set_title(f"{seq} resized\nmissing")
        ax[0, i].axis("off")

        # ---- normalized
        if norm_data[seq] is not None and z < norm_data[seq].shape[0]:
            img = norm_data[seq][z]
            ax[1, i].imshow(img, cmap="gray", interpolation="nearest")
            ax[1, i].set_title(
                f"{seq} normalized\n"
                f"min={img.min():.3f} max={img.max():.3f}\n"
                f"{norm_data[seq].shape}"
            )
        else:
            ax[1, i].set_title(f"{seq} normalized\nmissing")
        ax[1, i].axis("off")

    plt.suptitle(f"{patient_id} | {accession} | slice {z}", fontsize=14)
    plt.tight_layout()
    plt.show()

interact(
    show_slice,
    z=IntSlider(min=0, max=max_depth-1, step=1, value=max_depth//2)
);

interactive(children=(IntSlider(value=4, description='z', max=7), Output()), _dom_classes=('widget-interact',)…

## Deep Feature Extraction (3D CNN + Attention model)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# =========================
# CONFIG
# =========================
NORM_DIR = "/content/drive/MyDrive/Lung_Normalized_ROI"
SAVE_CSV = "/content/drive/MyDrive/lung_deep_features.csv"

MODALITIES = ["t1ce", "flair"]
TARGET_DEPTH = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

Using device: cpu


In [ ]:
def load_modalities(patient_id, accession, norm_dir, modalities):
    vols = {}
    reference_shape = None

    # Step 1: find any available modality
    for mod in modalities:
        path = os.path.join(norm_dir, f"{patient_id}_{accession}_{mod}_resized.npy")
        if os.path.exists(path):
            ref = np.load(path).astype(np.float32)
            reference_shape = ref.shape
            vols[mod] = ref
            break

    # If nothing exists → skip patient
    if reference_shape is None:
        raise ValueError(f"No modalities found for {patient_id} | {accession}")

    # Step 2: load or fill missing
    for mod in modalities:
        if mod in vols:
            continue

        path = os.path.join(norm_dir, f"{patient_id}_{accession}_{mod}_resized.npy")

        if os.path.exists(path):
            vols[mod] = np.load(path).astype(np.float32)
        else:
            print(f"⚠️ Missing {mod} → filling zeros for {patient_id} | {accession}")
            vols[mod] = np.zeros(reference_shape, dtype=np.float32)

    return vols

def get_center_slice_from_tumor(volume):
    nonzero_per_slice = np.count_nonzero(volume > 0, axis=(1, 2))
    return int(np.argmax(nonzero_per_slice))

def pad_or_crop_depth(volume, target_depth, center_slice):
    z, h, w = volume.shape
    out = np.zeros((target_depth, h, w), dtype=np.float32)

    if z == 0:
        return out

    center_slice = max(0, min(center_slice, z - 1))

    start = center_slice - target_depth // 2
    end = start + target_depth

    src_start = max(start, 0)
    src_end = min(end, z)

    if src_start >= src_end:
        return out

    dst_start = max(0, -start)
    length = src_end - src_start
    dst_end = dst_start + length

    out[dst_start:dst_end] = volume[src_start:src_end]
    return out

def build_3d_tensor(patient_id, accession, norm_dir, modalities, target_depth=64):
    vols = load_modalities(patient_id, accession, norm_dir, modalities)

    # anchor center from t1ce
    center_slice = get_center_slice_from_tumor(vols["t1ce"])

    channels = []
    for mod in modalities:
        vol = vols[mod]

        # clamp center slice to this modality's valid range
        center_mod = min(center_slice, vol.shape[0] - 1)

        vol_fixed = pad_or_crop_depth(vol, target_depth, center_mod)
        channels.append(vol_fixed)

    x = np.stack(channels, axis=0)   # (2, D, 256, 256)
    return torch.tensor(x, dtype=torch.float32), center_slice


In [ ]:
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = F.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w)).view(b, c, 1, 1, 1)
        return x * w

class CNN3D_Attention(nn.Module):
    def __init__(self, in_channels=2, num_classes=4, feature_dim=256):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.attn1 = SEBlock3D(32)

        self.block3 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.attn2 = SEBlock3D(64)

        self.block4 = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )

        self.feature_layer = nn.Linear(128, feature_dim)
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x, return_features=False):
        x = self.block1(x)
        x = self.block2(x)
        x = self.attn1(x)
        x = self.block3(x)
        x = self.attn2(x)
        x = self.block4(x)

        x = x.view(x.size(0), -1)
        features = self.feature_layer(x)

        if return_features:
            return features

        logits = self.classifier(features)
        return logits

In [ ]:
model = CNN3D_Attention(in_channels=2, num_classes=4, feature_dim=256)
model = model.to(DEVICE)
model.eval()

print("Lung 3D CNN + attention model ready.")

Lung 3D CNN + attention model ready.


In [ ]:
import re

files = sorted([f for f in os.listdir(NORM_DIR) if f.endswith(".npy")])

pairs = set()

pattern = re.compile(
    r"^(YG_[A-Z0-9]+)_(YG_[A-Z0-9]+)_(t1ce|flair)_resized\.npy$"
)

for f in files:
    m = pattern.match(f)
    if m:
        patient_id = m.group(1)
        accession = m.group(2)
        pairs.add((patient_id, accession))

pairs = sorted(list(pairs))

print("Total patient-accession pairs found:", len(pairs))
print("First 5 pairs:", pairs[:5])

Total patient-accession pairs found: 111
First 5 pairs: [('YG_0427RC24FT6W', 'YG_XKGYFGZHUMTO'), ('YG_0AXGKD8AFJGS', 'YG_HAEAKFOSOSWC'), ('YG_0IBUXTBINCD9', 'YG_985XO5NHL516'), ('YG_0LCI9NV44UK0', 'YG_WNJIJZXQW7BT'), ('YG_1BZ1NP805Q9J', 'YG_3ULZIC6OE5NB')]


testing a patient

In [ ]:
patient_id, accession = pairs[0]

x, center_slice = build_3d_tensor(
    patient_id,
    accession,
    NORM_DIR,
    MODALITIES,
    target_depth=TARGET_DEPTH
)

print("Patient:", patient_id)
print("Accession:", accession)
print("Center slice:", center_slice)
print("Tensor shape:", x.shape)   # should be (2, 64, 256, 256)

Patient: YG_0427RC24FT6W
Accession: YG_XKGYFGZHUMTO
Center slice: 2
Tensor shape: torch.Size([2, 64, 256, 256])


In [ ]:
with torch.no_grad():
    x_batch = x.unsqueeze(0).to(DEVICE)   # (1, 2, 64, 256, 256)
    features = model(x_batch, return_features=True)

print("Feature shape:", features.shape)   # should be (1, 256)
print("First 10 deep features:", features[0][:10])

Feature shape: torch.Size([1, 256])
First 10 deep features: tensor([ 0.0647, -0.0273,  0.0782, -0.0535,  0.0354, -0.0533,  0.0834, -0.0779,
        -0.0898, -0.0505])


All patients

In [ ]:
rows = []

with torch.no_grad():
    for i, (patient_id, accession) in enumerate(pairs, 1):
        try:
            x, center_slice = build_3d_tensor(
                patient_id,
                accession,
                NORM_DIR,
                MODALITIES,
                target_depth=TARGET_DEPTH
            )

            x_batch = x.unsqueeze(0).to(DEVICE)
            features = model(x_batch, return_features=True).cpu().numpy()[0]

            row = {
                "patient_id": patient_id,
                "accession": accession,
                "center_slice": center_slice
            }

            for j, val in enumerate(features):
                row[f"deep_f{j}"] = val

            rows.append(row)
            print(f"[{i}/{len(pairs)}] ✅ {patient_id} | {accession}")

        except Exception as e:
            print(f"[{i}/{len(pairs)}] ❌ Skipped {patient_id} | {accession}: {e}")

[1/111] ✅ YG_0427RC24FT6W | YG_XKGYFGZHUMTO
[2/111] ✅ YG_0AXGKD8AFJGS | YG_HAEAKFOSOSWC
[3/111] ✅ YG_0IBUXTBINCD9 | YG_985XO5NHL516
[4/111] ✅ YG_0LCI9NV44UK0 | YG_WNJIJZXQW7BT
[5/111] ✅ YG_1BZ1NP805Q9J | YG_3ULZIC6OE5NB
[6/111] ✅ YG_1PHFOPJWEXFN | YG_6U4LW891P2JA
[7/111] ✅ YG_1PHFOPJWEXFN | YG_U8D90FYR7XQ6
[8/111] ✅ YG_29KZUMN7VC2P | YG_QDPH4L1OB5Q3
[9/111] ✅ YG_2GBL4CHD8WHX | YG_MGO4964VSKLW
[10/111] ✅ YG_2XR67085OZD3 | YG_PT6LAWSXHST3
[11/111] ✅ YG_319S0V0DN4BI | YG_XY6SP25JK2UJ
⚠️ Missing t1ce → filling zeros for YG_3BA252G5NYX1 | YG_W7KRO4CZBDJ3
[12/111] ✅ YG_3BA252G5NYX1 | YG_W7KRO4CZBDJ3
[13/111] ✅ YG_3XK3D2ZF8DMH | YG_NHXKSO7LZ3OZ
[14/111] ✅ YG_496GIWWLVB29 | YG_E0SJA5ZEGI6T
[15/111] ✅ YG_4C0NA9IRB1QB | YG_748ADE23A96L
[16/111] ✅ YG_4MLUYTIFQUBW | YG_MGRE5V4Q0I2R
[17/111] ✅ YG_4MLUYTIFQUBW | YG_ZBH4LHYK7KVT
[18/111] ✅ YG_577MQ5KBZS0H | YG_OONB74IHNGUI
[19/111] ✅ YG_5AHF9B8AY5XH | YG_0CBM148C1MFN
[20/111] ✅ YG_5BOHJBD69BE2 | YG_LDY21C4TSC7L
[21/111] ✅ YG_5E1M7C728HGS | YG_34W2PP4

In [ ]:
df_deep = pd.DataFrame(rows)
df_deep.to_csv(SAVE_CSV, index=False)

print("✅ Lung deep features saved to:", SAVE_CSV)
print(df_deep.head())
print("Shape:", df_deep.shape)

✅ Lung deep features saved to: /content/drive/MyDrive/lung_deep_features.csv
        patient_id        accession  center_slice   deep_f0   deep_f1  \
0  YG_0427RC24FT6W  YG_XKGYFGZHUMTO             2  0.064695 -0.027321   
1  YG_0AXGKD8AFJGS  YG_HAEAKFOSOSWC            15  0.064557 -0.027275   
2  YG_0IBUXTBINCD9  YG_985XO5NHL516            79  0.064616 -0.027312   
3  YG_0LCI9NV44UK0  YG_WNJIJZXQW7BT             2  0.064637 -0.027370   
4  YG_1BZ1NP805Q9J  YG_3ULZIC6OE5NB             1  0.064712 -0.027347   

    deep_f2   deep_f3   deep_f4   deep_f5   deep_f6  ...  deep_f246  \
0  0.078248 -0.053536  0.035436 -0.053342  0.083409  ...   0.020886   
1  0.078427 -0.053492  0.035407 -0.053350  0.083546  ...   0.020632   
2  0.078380 -0.053508  0.035419 -0.053437  0.083505  ...   0.020686   
3  0.078284 -0.053481  0.035403 -0.053366  0.083460  ...   0.020874   
4  0.078256 -0.053528  0.035392 -0.053317  0.083410  ...   0.020864   

   deep_f247  deep_f248  deep_f249  deep_f250  deep_f251 

## Combining All Datasets

In [ ]:
# Step 1: Import pandas
import pandas as pd

# Step 2: Set file paths
clinical_file = "/content/drive/MyDrive/Brain-Mets-Lung-MRI-Path-Segs_Clinical_Data_2025_11_13.xlsx"
radiomics_file = "/content/drive/MyDrive/lung_radiomics_from_images.csv"
deep_file = "/content/drive/MyDrive/lung_deep_features.csv"

# Step 3: Load datasets
clinical_df = pd.read_excel(clinical_file, sheet_name="Clinical_Data")
radiomics_df = pd.read_csv(radiomics_file)
deep_df = pd.read_csv(deep_file)

# Step 4: Standardize columns for merge
clinical_df = clinical_df.rename(columns={"Accession": "accession", "Patient ID": "patient_id"})
radiomics_df = radiomics_df.rename(columns={"accession": "accession", "patient_ID": "patient_id"})
deep_df = deep_df.rename(columns={"accession": "accession", "patient_id": "patient_id"})

# Step 5: Clean the values
for df in [clinical_df, radiomics_df, deep_df]:
    df['accession'] = df['accession'].astype(str).str.strip()
    df['patient_id'] = df['patient_id'].astype(str).str.strip()

# Step 6: Merge using both keys
merged_df = pd.merge(clinical_df, radiomics_df, on=["accession", "patient_id"], how="inner")
merged_df = pd.merge(merged_df, deep_df, on=["accession", "patient_id"], how="inner")

# Step 7: Reorder columns: patient_id first, accession second
cols = merged_df.columns.tolist()
cols.remove("patient_id")
cols.remove("accession")
merged_df = merged_df[["patient_id", "accession"] + cols]

# Step 8: Save merged dataset back to Drive
merged_df.to_excel("/content/drive/MyDrive/BrainMetsLung_Dataset.xlsx", index=False)

print("Merged dataset shape:", merged_df.shape)
print("Merged dataset saved to your Drive!")

Merged dataset shape: (111, 489)
Merged dataset saved to your Drive!


## Classification

In [ ]:
!pip -q install imbalanced-learn xgboost lightgbm

In [ ]:
import os
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
# =========================
# LOAD DATA
# =========================

DATA_PATH = "/content/drive/MyDrive/BrainMetsLung_Dataset.xlsx"
TARGET_COL = "Histologic Subtype"
ID_COLS = ["patient_id", "accession"]
LEAKAGE_COLS = []
K_BEST = 100
MIN_COUNT = 3

ext = os.path.splitext(DATA_PATH)[1].lower()

if ext in [".xlsx", ".xls"]:
    xls = pd.ExcelFile(DATA_PATH)
    print("Sheets:", xls.sheet_names)
    df = pd.read_excel(DATA_PATH, sheet_name=xls.sheet_names[0])
elif ext == ".csv":
    try:
        df = pd.read_csv(DATA_PATH, sep=None, engine="python", encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, sep=None, engine="python", encoding="latin1")
else:
    raise ValueError(f"Unsupported file type: {ext}")

print("✅ Loaded dataset shape:", df.shape)

Sheets: ['Sheet1']
✅ Loaded dataset shape: (111, 489)


In [ ]:
# =========================
# CLEAN TARGET
# =========================
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(str)

# merge LUAD into Adenocarcinoma
df.loc[df[TARGET_COL] == "LUAD", TARGET_COL] = "Adenocarcinoma"

# group rare classes into Other
counts = df[TARGET_COL].value_counts()
rare_classes = counts[counts < MIN_COUNT].index
df.loc[df[TARGET_COL].isin(rare_classes), TARGET_COL] = "Other"

print("✅ Grouped into 'Other':", len(rare_classes), "classes")
print("New class counts:\n", df[TARGET_COL].value_counts())


✅ Grouped into 'Other': 7 classes
New class counts:
 Histologic Subtype
Adenocarcinoma    77
SCLC              12
NSCLC              8
Other              8
LCNEC              6
Name: count, dtype: int64


In [ ]:
# =========================
# DEFINE X / y
# =========================
X = df.drop(columns=[TARGET_COL] + ID_COLS + LEAKAGE_COLS, errors="ignore").copy()
y_raw = df[TARGET_COL].astype(str)

# convert numeric-looking strings safely
for c in X.columns:
    if X[c].dtype == "object":
        X[c] = X[c].astype(str).str.replace(",", ".", regex=False)
        X[c] = pd.to_numeric(X[c], errors="ignore")

le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Num classes:", len(le.classes_))
print("Classes:", le.classes_)


Num classes: 5
Classes: ['Adenocarcinoma' 'LCNEC' 'NSCLC' 'Other' 'SCLC']


/tmp/ipykernel_699/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_699/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_699/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_699/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_699/30514

In [ ]:
# =========================
# TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)


Train size: (88, 486) Test size: (23, 486)


In [ ]:
# =========================
# COLUMN TYPES
# =========================
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

# force categorical columns to string
X_train = X_train.copy()
X_test = X_test.copy()
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)


In [ ]:
# =========================
# PREPROCESSING
# =========================
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

# fit preprocess on train only
X_train_p = preprocess.fit_transform(X_train)
X_test_p = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()
K = min(K_BEST, X_train_p.shape[1])

selector = SelectKBest(score_func=mutual_info_classif, k=K)
X_train_sel = selector.fit_transform(X_train_p, y_train)
X_test_sel = selector.transform(X_test_p)

selected_idx = selector.get_support(indices=True)
selected_features = feature_names[selected_idx]

print("✅ Preprocessed features:", X_train_p.shape[1])
print(f"✅ Selected top K={K} features:", X_train_sel.shape[1])

selected_feat_path = "/content/drive/MyDrive/selected_top_features.txt"
with open(selected_feat_path, "w") as f:
    f.write("\n".join(selected_features))
print("✅ Saved selected feature list to:", selected_feat_path)


✅ Preprocessed features: 552
✅ Selected top K=100 features: 100
✅ Saved selected feature list to: /content/drive/MyDrive/selected_top_features.txt


In [ ]:
# =========================
# RESAMPLER CHOICE
# =========================
counts_train = Counter(y_train)
min_class_train = min(counts_train.values())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calculate minimum samples in a CV training fold for the minority class
# This assumes stratified splitting.
min_samples_in_cv_train_fold = min_class_train
if cv.n_splits > 0: # Avoid division by zero if cv.n_splits is somehow 0
    min_samples_in_cv_train_fold = min_class_train * (cv.n_splits - 1) // cv.n_splits

# SMOTE needs at least 2 samples per class to find neighbors
# and k_neighbors must be strictly less than the number of samples in the minority class in the fold
if min_samples_in_cv_train_fold < 2:
    resampler = RandomOverSampler(random_state=42)
    print("✅ Using RandomOverSampler (too few samples in CV folds for SMOTE)")
else:
    # k_neighbors for SMOTE must be strictly less than min_samples_in_cv_train_fold
    # Explicitly cap k_neighbors at a small value (e.g., 2) to prevent issues with very small minority classes
    k_neighbors = max(1, min(min_samples_in_cv_train_fold - 1, 2))
    resampler = SMOTE(random_state=42, k_neighbors=k_neighbors)
    print(f"✅ Using SMOTE with k_neighbors={k_neighbors} (based on min {min_samples_in_cv_train_fold} samples in CV folds)")


✅ Using SMOTE with k_neighbors=2 (based on min 4 samples in CV folds)


In [ ]:
# =========================
# MODELS
# =========================
n_classes = len(le.classes_)

models = {
    "LogReg": LogisticRegression(
        max_iter=5000,
        class_weight="balanced"
    ),
    "SVM_linear": SVC(
        kernel="linear",
        class_weight="balanced",
        probability=True
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        class_weight="balanced_subsample"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob" if n_classes > 2 else "binary:logistic",
        num_class=n_classes if n_classes > 2 else None,
        eval_metric="mlogloss" if n_classes > 2 else "logloss",
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800,
        learning_rate=0.03,
        random_state=42,
        verbose=-1
    )
}


In [ ]:

# =========================
# 5-FOLD CV + FINAL TEST
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

# binarized test labels for per-class AUC later
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

for name, model in models.items():
    print("\n" + "=" * 70)
    print("MODEL:", name)
    print("=" * 70)

    # full no-leakage pipeline for CV
    full_pipe = ImbPipeline(steps=[
        ("resample", resampler),
        ("model", model)
    ])

    # CV on TRAIN ONLY
    cv_scores = cross_val_score(
        full_pipe,
        X_train_sel,
        y_train,
        cv=cv,
        scoring="roc_auc_ovr",
        n_jobs=-1
    )

    print("CV ROC-AUC scores:", np.round(cv_scores, 4))
    print("Mean CV ROC-AUC:", round(np.mean(cv_scores), 4))

    # fit final model on full train
    X_train_bal, y_train_bal = resampler.fit_resample(X_train_sel, y_train)
    model.fit(X_train_bal, y_train_bal)

    y_pred = model.predict(X_test_sel)
    y_score = model.predict_proba(X_test_sel)

    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average="macro")
    auc_macro = roc_auc_score(y_test, y_score, multi_class="ovr", average="macro")

    print("\nBalanced Accuracy:", round(bal_acc, 4))
    print("F1 Macro:", round(f1m, 4))
    print("Test ROC-AUC (macro):", round(auc_macro, 4))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    # per-class AUC
    print("\nPer-class ROC-AUC:")
    per_class_auc = {}
    for i, cls_name in enumerate(le.classes_):
        try:
            auc_i = roc_auc_score(y_test_bin[:, i], y_score[:, i])
            per_class_auc[cls_name] = auc_i
            print(f"{cls_name}: {auc_i:.4f}")
        except:
            per_class_auc[cls_name] = np.nan
            print(f"{cls_name}: AUC not defined")

    results.append({
        "Model": name,
        "CV_ROC_AUC_Mean": np.mean(cv_scores),
        "Test_ROC_AUC_Macro": auc_macro,
        "BalancedAccuracy": bal_acc,
        "F1_macro": f1m
    })


MODEL: LogReg
CV ROC-AUC scores: [0.698  0.6745 0.8696 0.7375 0.7425]
Mean CV ROC-AUC: 0.7444

Balanced Accuracy: 0.7
F1 Macro: 0.7216
Test ROC-AUC (macro): 0.9078

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.89      1.00      0.94        16
         LCNEC       0.00      0.00      0.00         1
         NSCLC       1.00      1.00      1.00         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.91        23
     macro avg       0.78      0.70      0.72        23
  weighted avg       0.88      0.91      0.89        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 1  0  0  0  0]
 [ 0  0  2  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9375
LCNEC: 0.8636
NSCLC: 1.0000
Other: 0.7381
SCLC: 1.0000

MODEL: SVM_linear


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CV ROC-AUC scores: [0.855  0.8261 0.9577 0.8    0.7267]
Mean CV ROC-AUC: 0.8331

Balanced Accuracy: 0.6
F1 Macro: 0.6549
Test ROC-AUC (macro): 0.9219

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.89      1.00      0.94        16
         LCNEC       0.00      0.00      0.00         1
         NSCLC       1.00      0.50      0.67         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.87        23
     macro avg       0.78      0.60      0.65        23
  weighted avg       0.88      0.87      0.86        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 1  0  0  0  0]
 [ 0  1  1  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9643
LCNEC: 0.9545
NSCLC: 1.0000
Other: 0.6905
SCLC: 1.0000

MODEL: RandomForest
CV ROC-AUC scores: [0.9221 0.9079 0.9382 0.8    0.9496]
Mean CV ROC-AUC: 0.9036

Balanced 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CV ROC-AUC scores: [0.9882 0.9757 0.9452 0.9125 0.9125]
Mean CV ROC-AUC: 0.9468

Balanced Accuracy: 0.9
F1 Macro: 0.9273
Test ROC-AUC (macro): 0.9089

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.94      1.00      0.97        16
         LCNEC       1.00      1.00      1.00         1
         NSCLC       1.00      1.00      1.00         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.96        23
     macro avg       0.99      0.90      0.93        23
  weighted avg       0.96      0.96      0.95        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 0  1  0  0  0]
 [ 0  0  2  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9018
LCNEC: 1.0000
NSCLC: 1.0000
Other: 0.6429
SCLC: 1.0000

MODEL: LightGBM
CV ROC-AUC scores: [0.9765 0.964  0.9757 0.9625 0.9375]
Mean CV ROC-AUC: 0.9632

Balanced Accu

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [ ]:
# =========================
# SUMMARY TABLE
# =========================
results_df = pd.DataFrame(results).sort_values(
    by=["CV_ROC_AUC_Mean", "Test_ROC_AUC_Macro"],
    ascending=False
)

print("\n✅ Model comparison:")
print(results_df)


✅ Model comparison:
          Model  CV_ROC_AUC_Mean  Test_ROC_AUC_Macro  BalancedAccuracy  \
4      LightGBM         0.963235            0.926190               0.7   
3       XGBoost         0.946838            0.908929               0.9   
2  RandomForest         0.903556            0.864935               0.6   
1    SVM_linear         0.833092            0.921861               0.6   
0        LogReg         0.744440            0.907846               0.7   

   F1_macro  
4  0.687273  
3  0.927273  
2  0.649524  
1  0.654902  
0  0.721569  


## Classification 2

In [ ]:
!pip -q install imbalanced-learn xgboost lightgbm

In [ ]:
import os
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
# =========================
# LOAD DATA
# =========================

DATA_PATH = "/content/drive/MyDrive/BrainMetsLung_Dataset.xlsx"
TARGET_COL = "Histologic Subtype"
ID_COLS = ["patient_id", "accession"]
LEAKAGE_COLS = []
K_BEST = 100
MIN_COUNT = 3

ext = os.path.splitext(DATA_PATH)[1].lower()

if ext in [".xlsx", ".xls"]:
    xls = pd.ExcelFile(DATA_PATH)
    print("Sheets:", xls.sheet_names)
    df = pd.read_excel(DATA_PATH, sheet_name=xls.sheet_names[0])
elif ext == ".csv":
    try:
        df = pd.read_csv(DATA_PATH, sep=None, engine="python", encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_PATH, sep=None, engine="python", encoding="latin1")
else:
    raise ValueError(f"Unsupported file type: {ext}")

print("✅ Loaded dataset shape:", df.shape)

Sheets: ['Sheet1']
✅ Loaded dataset shape: (111, 489)


In [ ]:
# =========================
# CLEAN TARGET
# =========================
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(str)

# merge LUAD into Adenocarcinoma
df.loc[df[TARGET_COL] == "LUAD", TARGET_COL] = "Adenocarcinoma"

# group rare classes into Other
counts = df[TARGET_COL].value_counts()
rare_classes = counts[counts < MIN_COUNT].index
df.loc[df[TARGET_COL].isin(rare_classes), TARGET_COL] = "Other"

print("✅ Grouped into 'Other':", len(rare_classes), "classes")
print("New class counts:\n", df[TARGET_COL].value_counts())


✅ Grouped into 'Other': 7 classes
New class counts:
 Histologic Subtype
Adenocarcinoma    77
SCLC              12
NSCLC              8
Other              8
LCNEC              6
Name: count, dtype: int64


In [ ]:
# =========================
# DEFINE X / y
# =========================
X = df.drop(columns=[TARGET_COL] + ID_COLS + LEAKAGE_COLS, errors="ignore").copy()
y_raw = df[TARGET_COL].astype(str)

# convert numeric-looking strings safely
for c in X.columns:
    if X[c].dtype == "object":
        X[c] = X[c].astype(str).str.replace(",", ".", regex=False)
        X[c] = pd.to_numeric(X[c], errors="ignore")

le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Num classes:", len(le.classes_))
print("Classes:", le.classes_)


Num classes: 5
Classes: ['Adenocarcinoma' 'LCNEC' 'NSCLC' 'Other' 'SCLC']


/tmp/ipykernel_1376/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_1376/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_1376/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_1376/3051474050.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X[c] = pd.to_numeric(X[c], errors="ignore")
/tmp/ipykernel_1376/

In [ ]:
# =========================
# TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)


Train size: (88, 486) Test size: (23, 486)


In [ ]:
# =========================
# COLUMN TYPES
# =========================
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

# force categorical columns to string
X_train = X_train.copy()
X_test = X_test.copy()
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)


In [ ]:
# =========================
# PREPROCESSING
# =========================
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

# fit preprocess on train only
X_train_p = preprocess.fit_transform(X_train)
X_test_p = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()
K = min(K_BEST, X_train_p.shape[1])

selector = SelectKBest(score_func=mutual_info_classif, k=K)
X_train_sel = selector.fit_transform(X_train_p, y_train)
X_test_sel = selector.transform(X_test_p)

selected_idx = selector.get_support(indices=True)
selected_features = feature_names[selected_idx]

print("✅ Preprocessed features:", X_train_p.shape[1])
print(f"✅ Selected top K={K} features:", X_train_sel.shape[1])

selected_feat_path = "/content/drive/MyDrive/selected_top_features.txt"
with open(selected_feat_path, "w") as f:
    f.write("\n".join(selected_features))
print("✅ Saved selected feature list to:", selected_feat_path)


✅ Preprocessed features: 552
✅ Selected top K=100 features: 100
✅ Saved selected feature list to: /content/drive/MyDrive/selected_top_features.txt


In [ ]:
# =========================
# RESAMPLER CHOICE
# =========================
counts_train = Counter(y_train)
min_class_train = min(counts_train.values())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calculate minimum samples in a CV training fold for the minority class
# This assumes stratified splitting.
min_samples_in_cv_train_fold = min_class_train
if cv.n_splits > 0: # Avoid division by zero if cv.n_splits is somehow 0
    min_samples_in_cv_train_fold = min_class_train * (cv.n_splits - 1) // cv.n_splits

# SMOTE needs at least 2 samples per class to find neighbors
# and k_neighbors must be strictly less than the number of samples in the minority class in the fold
if min_samples_in_cv_train_fold < 2:
    resampler = RandomOverSampler(random_state=42)
    print("✅ Using RandomOverSampler (too few samples in CV folds for SMOTE)")
else:
    # k_neighbors for SMOTE must be strictly less than min_samples_in_cv_train_fold
    # Explicitly cap k_neighbors at a small value (e.g., 2) to prevent issues with very small minority classes
    k_neighbors = max(1, min(min_samples_in_cv_train_fold - 1, 2))
    resampler = SMOTE(random_state=42, k_neighbors=k_neighbors)
    print(f"✅ Using SMOTE with k_neighbors={k_neighbors} (based on min {min_samples_in_cv_train_fold} samples in CV folds)")


✅ Using SMOTE with k_neighbors=2 (based on min 4 samples in CV folds)


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# =========================
# MODELS
# =========================
n_classes = len(le.classes_)

models = {
    "LogReg": LogisticRegression(
        max_iter=5000,
        class_weight="balanced"
    ),

    "SVM_linear": SVC(
        kernel="linear",
        class_weight="balanced",
        probability=True
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        class_weight="balanced_subsample"
    ),

    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob" if n_classes > 2 else "binary:logistic",
        num_class=n_classes if n_classes > 2 else None,
        eval_metric="mlogloss" if n_classes > 2 else "logloss",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=800,
        learning_rate=0.03,
        random_state=42,
        verbose=-1
    )
}

In [ ]:

# =========================
# 5-FOLD CV + FINAL TEST
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

# binarized test labels for per-class AUC later
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

for name, model in models.items():
    print("\n" + "=" * 70)
    print("MODEL:", name)
    print("=" * 70)

    # full no-leakage pipeline for CV
    full_pipe = ImbPipeline(steps=[
        ("resample", resampler),
        ("model", model)
    ])

    # CV on TRAIN ONLY
    cv_scores = cross_val_score(
        full_pipe,
        X_train_sel,
        y_train,
        cv=cv,
        scoring="roc_auc_ovr",
        n_jobs=-1
    )

    print("CV ROC-AUC scores:", np.round(cv_scores, 4))
    print("Mean CV ROC-AUC:", round(np.mean(cv_scores), 4))

    # fit final model on full train
    X_train_bal, y_train_bal = resampler.fit_resample(X_train_sel, y_train)
    model.fit(X_train_bal, y_train_bal)

    y_pred = model.predict(X_test_sel)
    y_score = model.predict_proba(X_test_sel)

    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average="macro")
    auc_macro = roc_auc_score(y_test, y_score, multi_class="ovr", average="macro")

    print("\nBalanced Accuracy:", round(bal_acc, 4))
    print("F1 Macro:", round(f1m, 4))
    print("Test ROC-AUC (macro):", round(auc_macro, 4))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    # per-class AUC
    print("\nPer-class ROC-AUC:")
    per_class_auc = {}
    for i, cls_name in enumerate(le.classes_):
        try:
            auc_i = roc_auc_score(y_test_bin[:, i], y_score[:, i])
            per_class_auc[cls_name] = auc_i
            print(f"{cls_name}: {auc_i:.4f}")
        except:
            per_class_auc[cls_name] = np.nan
            print(f"{cls_name}: AUC not defined")

    results.append({
        "Model": name,
        "CV_ROC_AUC_Mean": np.mean(cv_scores),
        "Test_ROC_AUC_Macro": auc_macro,
        "BalancedAccuracy": bal_acc,
        "F1_macro": f1m
    })


MODEL: LogReg
CV ROC-AUC scores: [0.7316 0.7243 0.8634 0.75   0.7425]
Mean CV ROC-AUC: 0.7624

Balanced Accuracy: 0.7
F1 Macro: 0.7216
Test ROC-AUC (macro): 0.9152

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.89      1.00      0.94        16
         LCNEC       0.00      0.00      0.00         1
         NSCLC       1.00      1.00      1.00         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.91        23
     macro avg       0.78      0.70      0.72        23
  weighted avg       0.88      0.91      0.89        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 1  0  0  0  0]
 [ 0  0  2  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9286
LCNEC: 0.9091
NSCLC: 1.0000
Other: 0.7381
SCLC: 1.0000

MODEL: SVM_linear


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CV ROC-AUC scores: [0.8785 0.8385 0.9515 0.8    0.7667]
Mean CV ROC-AUC: 0.847

Balanced Accuracy: 0.6
F1 Macro: 0.6549
Test ROC-AUC (macro): 0.9284

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.89      1.00      0.94        16
         LCNEC       0.00      0.00      0.00         1
         NSCLC       1.00      0.50      0.67         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.87        23
     macro avg       0.78      0.60      0.65        23
  weighted avg       0.88      0.87      0.86        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 1  0  0  0  0]
 [ 0  1  1  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9732
LCNEC: 0.9545
NSCLC: 1.0000
Other: 0.7143
SCLC: 1.0000

MODEL: RandomForest
CV ROC-AUC scores: [0.9134 0.8636 0.9007 0.75   0.9558]
Mean CV ROC-AUC: 0.8767

Balanced A

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CV ROC-AUC scores: [0.9412 0.9321 0.9452 0.925  0.925 ]
Mean CV ROC-AUC: 0.9337

Balanced Accuracy: 0.9
F1 Macro: 0.9273
Test ROC-AUC (macro): 0.9524

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.94      1.00      0.97        16
         LCNEC       1.00      1.00      1.00         1
         NSCLC       1.00      1.00      1.00         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.96        23
     macro avg       0.99      0.90      0.93        23
  weighted avg       0.96      0.96      0.95        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 0  1  0  0  0]
 [ 0  0  2  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9286
LCNEC: 1.0000
NSCLC: 1.0000
Other: 0.8333
SCLC: 1.0000

MODEL: MLP
CV ROC-AUC scores: [0.8034 0.6273 0.8715 0.7142 0.6958]
Mean CV ROC-AUC: 0.7424

Balanced Accuracy:

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CV ROC-AUC scores: [0.9765 0.964  0.939  0.9125 0.9375]
Mean CV ROC-AUC: 0.9459

Balanced Accuracy: 0.9
F1 Macro: 0.9273
Test ROC-AUC (macro): 0.928

Classification Report:
                precision    recall  f1-score   support

Adenocarcinoma       0.94      1.00      0.97        16
         LCNEC       1.00      1.00      1.00         1
         NSCLC       1.00      1.00      1.00         2
         Other       1.00      0.50      0.67         2
          SCLC       1.00      1.00      1.00         2

      accuracy                           0.96        23
     macro avg       0.99      0.90      0.93        23
  weighted avg       0.96      0.96      0.95        23

Confusion Matrix:
 [[16  0  0  0  0]
 [ 0  1  0  0  0]
 [ 0  0  2  0  0]
 [ 1  0  0  1  0]
 [ 0  0  0  0  2]]

Per-class ROC-AUC:
Adenocarcinoma: 0.9018
LCNEC: 1.0000
NSCLC: 1.0000
Other: 0.7381
SCLC: 1.0000

MODEL: LightGBM
CV ROC-AUC scores: [0.9765 0.964  0.964  0.9375 0.925 ]
Mean CV ROC-AUC: 0.9534

Balanced Accur

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [ ]:
# =========================
# SUMMARY TABLE
# =========================
results_df = pd.DataFrame(results).sort_values(
    by=["CV_ROC_AUC_Mean", "Test_ROC_AUC_Macro"],
    ascending=False
)

print("\n✅ Model comparison:")
print(results_df)


✅ Model comparison:
              Model  CV_ROC_AUC_Mean  Test_ROC_AUC_Macro  BalancedAccuracy  \
6          LightGBM         0.953382            0.927976            0.8000   
5           XGBoost         0.945882            0.927976            0.9000   
3  GradientBoosting         0.933701            0.952381            0.9000   
2      RandomForest         0.876704            0.930249            0.6000   
1        SVM_linear         0.847040            0.928409            0.6000   
0            LogReg         0.762376            0.915152            0.7000   
4               MLP         0.742448            0.831385            0.3875   

   F1_macro  
6  0.753939  
5  0.927273  
3  0.927273  
2  0.649524  
1  0.654902  
0  0.721569  
4  0.384762  
